In [ ]:
!pip install -q \
  torch>=2.1.0 \
  transformers>=4.43.0 \
  accelerate>=0.30.0 \
  sentencepiece


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    use_cache=True,
    attn_implementation="eager"
)

model.eval()

inputs = tokenizer("Hello world", return_tensors="pt").to(model.device)

with torch.no_grad():
    out = model(
        **inputs,
        output_attentions=True,
        use_cache=True
    )

print("Number of layers:", len(out.attentions))
print("Attention shape (layer 0):", out.attentions[0].shape)

# -------------------------------------------------
# Universal Safe DynamicCache Access
# -------------------------------------------------
cache = out.past_key_values
print("Cache type:", type(cache))

layers = list(cache)
print("Number of cache layers:", len(layers))

layer0 = layers[0]
print("Layer 0 length:", len(layer0))

# Always take first two entries as key/value
k = layer0[0]
v = layer0[1]

print("Key shape (layer 0):", k.shape)
print("Value shape (layer 0):", v.shape)

# -------------------------------------------------
# Compute Total KV Memory
# -------------------------------------------------
total_elements = 0

for layer in layers:
    k = layer[0]
    v = layer[1]
    total_elements += k.numel() + v.numel()

bytes_per_element = 2 if model.dtype == torch.float16 else 4
total_memory_mb = total_elements * bytes_per_element / (1024 ** 2)

print("Total KV elements:", total_elements)
print(f"Total KV memory: {total_memory_mb:.6f} MB")

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Number of layers: 16
Attention shape (layer 0): torch.Size([1, 32, 3, 3])
Cache type: <class 'transformers.cache_utils.DynamicCache'>
Number of cache layers: 16
Layer 0 length: 3
Key shape (layer 0): torch.Size([1, 8, 3, 64])
Value shape (layer 0): torch.Size([1, 8, 3, 64])
Total KV elements: 49152
Total KV memory: 0.093750 MB


In [ ]:
import torch
from transformers.cache_utils import DynamicCache

# ----- KV expert definitions
class KVExpert:
    def update(self, past_kv, new_kv, layer_idx):
        raise NotImplementedError


class FullCacheExpert(KVExpert):
    def update(self, past_kv, new_kv, layer_idx):
        return new_kv


# ----- Inputs
inputs = tokenizer("Hello, I am Ahad. Who are you?", return_tensors="pt").to(model.device)
input_ids = inputs["input_ids"]

past_kv = DynamicCache()
expert = FullCacheExpert()

# ----- Autoregressive loop
for step in range(3):

    if past_kv.get_seq_length() == 0:
        step_input_ids = input_ids
    else:
        step_input_ids = input_ids[:, -1:]

    with torch.no_grad():
        out = model(
            input_ids=step_input_ids,
            past_key_values=past_kv,
            use_cache=True
        )

    next_token = out.logits[:, -1].argmax(dim=-1, keepdim=True)

    # Convert caches to lists (SAFE)
    new_layers = list(out.past_key_values)
    past_layers = list(past_kv)

    for layer_idx, layer in enumerate(new_layers):

        k = layer[0]
        v = layer[1]

        past_layer = past_layers[layer_idx] if layer_idx < len(past_layers) else None

        k_new, v_new = expert.update(
            past_layer,
            (k, v),
            layer_idx
        )

        past_kv.update(k_new, v_new, layer_idx)

    input_ids = torch.cat([input_ids, next_token], dim=-1)

    # debug
    print(f"\nStep {step}")
    updated_layers = list(past_kv)
    for i, layer in enumerate(updated_layers):
        k_i = layer[0]
        print(f"  Layer {i}: seq_len = {k_i.shape[2]}")


Step 0
  Layer 0: seq_len = 24
  Layer 1: seq_len = 24
  Layer 2: seq_len = 24
  Layer 3: seq_len = 24
  Layer 4: seq_len = 24
  Layer 5: seq_len = 24
  Layer 6: seq_len = 24
  Layer 7: seq_len = 24
  Layer 8: seq_len = 24
  Layer 9: seq_len = 24
  Layer 10: seq_len = 24
  Layer 11: seq_len = 24
  Layer 12: seq_len = 24
  Layer 13: seq_len = 24
  Layer 14: seq_len = 24
  Layer 15: seq_len = 24

Step 1
  Layer 0: seq_len = 50
  Layer 1: seq_len = 50
  Layer 2: seq_len = 50
  Layer 3: seq_len = 50
  Layer 4: seq_len = 50
  Layer 5: seq_len = 50
  Layer 6: seq_len = 50
  Layer 7: seq_len = 50
  Layer 8: seq_len = 50
  Layer 9: seq_len = 50
  Layer 10: seq_len = 50
  Layer 11: seq_len = 50
  Layer 12: seq_len = 50
  Layer 13: seq_len = 50
  Layer 14: seq_len = 50
  Layer 15: seq_len = 50

Step 2
  Layer 0: seq_len = 102
  Layer 1: seq_len = 102
  Layer 2: seq_len = 102
  Layer 3: seq_len = 102
  Layer 4: seq_len = 102
  Layer 5: seq_len = 102
  Layer 6: seq_len = 102
  Layer 7: seq_len = 

In [ ]:
import torch
from transformers.cache_utils import DynamicCache

# ----- KV expert definitions
class KVExpert:
    def update(self, past_kv_layer, new_kv_delta, layer_idx):
        raise NotImplementedError


class LayerSparseExpert(KVExpert):
    def __init__(self, drop_layers):
        self.drop_layers = set(drop_layers)

    def update(self, past_kv_layer, new_kv_delta, layer_idx):
        k_delta, v_delta = new_kv_delta

        if layer_idx in self.drop_layers:
            k_delta = torch.zeros_like(k_delta)
            v_delta = torch.zeros_like(v_delta)

        return k_delta, v_delta


# ----- Inputs
inputs = tokenizer("Hello", return_tensors="pt").to(model.device)
input_ids = inputs["input_ids"]

past_kv = DynamicCache()
expert = LayerSparseExpert(drop_layers=[0, 2, 4, 6])

# ----- Autoregressive loop
for step in range(3):

    old_len = past_kv.get_seq_length()

    step_input_ids = input_ids if old_len == 0 else input_ids[:, -1:]

    with torch.no_grad():
        out = model(
            input_ids=step_input_ids,
            past_key_values=past_kv,
            use_cache=True
        )

    next_token = out.logits[:, -1].argmax(dim=-1, keepdim=True)

    # Convert caches to lists (safe)
    new_layers = list(out.past_key_values)
    past_layers = list(past_kv)

    # ----- delta update only
    for layer_idx, layer in enumerate(new_layers):

        # Your HF version returns tuple of len 3
        k_full = layer[0]
        v_full = layer[1]

        # slice only NEW tokens
        k_delta = k_full[:, :, old_len:, :]
        v_delta = v_full[:, :, old_len:, :]

        past_layer = past_layers[layer_idx] if layer_idx < len(past_layers) else None

        k_upd, v_upd = expert.update(
            past_layer,
            (k_delta, v_delta),
            layer_idx
        )

        # update with delta only
        past_kv.update(k_upd, v_upd, layer_idx)

    input_ids = torch.cat([input_ids, next_token], dim=-1)

    # ----- debug
    print(f"\nStep {step}")
    layers = list(past_kv)
    for i, layer in enumerate(layers):
        print(f"  Layer {i}: seq_len = {layer[0].shape[2]}")


Step 0
  Layer 0: seq_len = 4
  Layer 1: seq_len = 4
  Layer 2: seq_len = 4
  Layer 3: seq_len = 4
  Layer 4: seq_len = 4
  Layer 5: seq_len = 4
  Layer 6: seq_len = 4
  Layer 7: seq_len = 4
  Layer 8: seq_len = 4
  Layer 9: seq_len = 4
  Layer 10: seq_len = 4
  Layer 11: seq_len = 4
  Layer 12: seq_len = 4
  Layer 13: seq_len = 4
  Layer 14: seq_len = 4
  Layer 15: seq_len = 4

Step 1
  Layer 0: seq_len = 6
  Layer 1: seq_len = 6
  Layer 2: seq_len = 6
  Layer 3: seq_len = 6
  Layer 4: seq_len = 6
  Layer 5: seq_len = 6
  Layer 6: seq_len = 6
  Layer 7: seq_len = 6
  Layer 8: seq_len = 6
  Layer 9: seq_len = 6
  Layer 10: seq_len = 6
  Layer 11: seq_len = 6
  Layer 12: seq_len = 6
  Layer 13: seq_len = 6
  Layer 14: seq_len = 6
  Layer 15: seq_len = 6

Step 2
  Layer 0: seq_len = 8
  Layer 1: seq_len = 8
  Layer 2: seq_len = 8
  Layer 3: seq_len = 8
  Layer 4: seq_len = 8
  Layer 5: seq_len = 8
  Layer 6: seq_len = 8
  Layer 7: seq_len = 8
  Layer 8: seq_len = 8
  Layer 9: seq_len = 

In [ ]:
class HeadSpecificExpert(KVExpert):
    def __init__(self, keep_heads, window=1):
        self.keep_heads = set(keep_heads)
        self.window = window

    def update(self, past_kv, new_kv, layer_idx):
        k, v = new_kv                      # (B, H, S, D)
        B, H, S, D = k.shape

        # clone to preserve shape, dtype, device
        k_new = k.clone()
        v_new = v.clone()

        # zero-out old tokens for non-kept heads
        for h in range(H):
            if h not in self.keep_heads:
                if S > self.window:
                    k_new[:, h, :-self.window, :] = 0
                    v_new[:, h, :-self.window, :] = 0

        # 🔒 dtype safety (critical)
        k_new = k_new.to(k.dtype)
        v_new = v_new.to(v.dtype)

        return k_new, v_new


In [ ]:
inputs = tokenizer("Hello", return_tensors="pt").to(model.device)
input_ids = inputs["input_ids"]

past_kv = DynamicCache()
expert = HeadSpecificExpert(keep_heads=[0], window=1)

for step in range(3):

    old_len = past_kv.get_seq_length()
    step_input_ids = input_ids if old_len == 0 else input_ids[:, -1:]

    with torch.no_grad():
        out = model(
            input_ids=step_input_ids,
            past_key_values=past_kv,
            use_cache=True
        )

    next_token = out.logits[:, -1].argmax(dim=-1, keepdim=True)

    new_layers = list(out.past_key_values)
    past_layers = list(past_kv)

    for layer_idx, layer in enumerate(new_layers):

        # your HF version returns (k, v, extra)
        k_full = layer[0]
        v_full = layer[1]

        # ---- DELTA ONLY (critical!)
        k_delta = k_full[:, :, old_len:, :]
        v_delta = v_full[:, :, old_len:, :]

        if layer_idx == 1:
            past_layer = past_layers[layer_idx] if layer_idx < len(past_layers) else None
            k_upd, v_upd = expert.update(
                past_layer,
                (k_delta, v_delta),
                layer_idx
            )
        else:
            k_upd, v_upd = k_delta, v_delta

        past_kv.update(k_upd, v_upd, layer_idx)

    input_ids = torch.cat([input_ids, next_token], dim=-1)

    # ---- Debug
    print(f"\nStep {step}")
    layers = list(past_kv)
    for i, layer in enumerate(layers):
        print(f"  Layer {i}: seq_len = {layer[0].shape[2]}")


Step 0
  Layer 0: seq_len = 4
  Layer 1: seq_len = 4
  Layer 2: seq_len = 4
  Layer 3: seq_len = 4
  Layer 4: seq_len = 4
  Layer 5: seq_len = 4
  Layer 6: seq_len = 4
  Layer 7: seq_len = 4
  Layer 8: seq_len = 4
  Layer 9: seq_len = 4
  Layer 10: seq_len = 4
  Layer 11: seq_len = 4
  Layer 12: seq_len = 4
  Layer 13: seq_len = 4
  Layer 14: seq_len = 4
  Layer 15: seq_len = 4

Step 1
  Layer 0: seq_len = 6
  Layer 1: seq_len = 6
  Layer 2: seq_len = 6
  Layer 3: seq_len = 6
  Layer 4: seq_len = 6
  Layer 5: seq_len = 6
  Layer 6: seq_len = 6
  Layer 7: seq_len = 6
  Layer 8: seq_len = 6
  Layer 9: seq_len = 6
  Layer 10: seq_len = 6
  Layer 11: seq_len = 6
  Layer 12: seq_len = 6
  Layer 13: seq_len = 6
  Layer 14: seq_len = 6
  Layer 15: seq_len = 6

Step 2
  Layer 0: seq_len = 8
  Layer 1: seq_len = 8
  Layer 2: seq_len = 8
  Layer 3: seq_len = 8
  Layer 4: seq_len = 8
  Layer 5: seq_len = 8
  Layer 6: seq_len = 8
  Layer 7: seq_len = 8
  Layer 8: seq_len = 8
  Layer 9: seq_len = 

In [ ]:
class TopKTokenExpert(KVExpert):
    def __init__(self, K=16):
        self.K = K

    def update(self, past_kv, new_kv, layer_idx):
        k, v = new_kv                  # (B, H, S, D)
        B, H, S, D = k.shape

        # clone to preserve shape, dtype, device
        k_new = k.clone()
        v_new = v.clone()

        if S > self.K:
            # zero out all but last K tokens
            k_new[:, :, :-self.K, :] = 0
            v_new[:, :, :-self.K, :] = 0

        # 🔒 dtype safety
        k_new = k_new.to(k.dtype)
        v_new = v_new.to(v.dtype)

        return k_new, v_new


In [ ]:
inputs = tokenizer("Hello", return_tensors="pt").to(model.device)
input_ids = inputs["input_ids"]

past_kv = DynamicCache()
expert = TopKTokenExpert(K=1)

for step in range(3):

    old_len = past_kv.get_seq_length()
    step_input_ids = input_ids if old_len == 0 else input_ids[:, -1:]

    with torch.no_grad():
        out = model(
            input_ids=step_input_ids,
            past_key_values=past_kv,
            use_cache=True
        )

    next_token = out.logits[:, -1].argmax(dim=-1, keepdim=True)

    new_layers = list(out.past_key_values)
    past_layers = list(past_kv)

    for layer_idx, layer in enumerate(new_layers):

        # your HF build returns (k, v, extra)
        k_full = layer[0]
        v_full = layer[1]

        # ---- delta only
        k_delta = k_full[:, :, old_len:, :]
        v_delta = v_full[:, :, old_len:, :]

        if layer_idx == 1:
            past_layer = past_layers[layer_idx] if layer_idx < len(past_layers) else None
            k_upd, v_upd = expert.update(
                past_layer,
                (k_delta, v_delta),
                layer_idx
            )
        else:
            k_upd, v_upd = k_delta, v_delta

        past_kv.update(k_upd, v_upd, layer_idx)

    input_ids = torch.cat([input_ids, next_token], dim=-1)

    # ---- Debug
    print(f"\nStep {step}")
    layers = list(past_kv)
    for i, layer in enumerate(layers):
        print(f"  Layer {i}: seq_len = {layer[0].shape[2]}")


Step 0
  Layer 0: seq_len = 4
  Layer 1: seq_len = 4
  Layer 2: seq_len = 4
  Layer 3: seq_len = 4
  Layer 4: seq_len = 4
  Layer 5: seq_len = 4
  Layer 6: seq_len = 4
  Layer 7: seq_len = 4
  Layer 8: seq_len = 4
  Layer 9: seq_len = 4
  Layer 10: seq_len = 4
  Layer 11: seq_len = 4
  Layer 12: seq_len = 4
  Layer 13: seq_len = 4
  Layer 14: seq_len = 4
  Layer 15: seq_len = 4

Step 1
  Layer 0: seq_len = 6
  Layer 1: seq_len = 6
  Layer 2: seq_len = 6
  Layer 3: seq_len = 6
  Layer 4: seq_len = 6
  Layer 5: seq_len = 6
  Layer 6: seq_len = 6
  Layer 7: seq_len = 6
  Layer 8: seq_len = 6
  Layer 9: seq_len = 6
  Layer 10: seq_len = 6
  Layer 11: seq_len = 6
  Layer 12: seq_len = 6
  Layer 13: seq_len = 6
  Layer 14: seq_len = 6
  Layer 15: seq_len = 6

Step 2
  Layer 0: seq_len = 8
  Layer 1: seq_len = 8
  Layer 2: seq_len = 8
  Layer 3: seq_len = 8
  Layer 4: seq_len = 8
  Layer 5: seq_len = 8
  Layer 6: seq_len = 8
  Layer 7: seq_len = 8
  Layer 8: seq_len = 8
  Layer 9: seq_len = 

In [ ]:
import torch

# ----- Token importance (last query, top-5%)
def token_importance(attn_weights):
    # attn_weights: (B, QH, Q, K)
    last_attn = attn_weights[:, :, -1, :]        # (B, QH, K)
    k = max(1, last_attn.shape[-1] // 20)
    topk_vals, _ = torch.topk(last_attn, k=k, dim=-1)
    return topk_vals.mean().item()


# ----- Attention entropy (averaged over heads, last query)
def attention_entropy(attn_weights):
    # attn_weights: (B, QH, Q, K)
    p = attn_weights[:, :, -1, :].mean(dim=1)    # (B, K)
    p = p / (p.sum(dim=-1, keepdim=True) + 1e-8) # normalize
    entropy = -(p * torch.log(p + 1e-8)).sum(dim=-1)
    return entropy.mean().item()


# ----- Count KV elements in DynamicCache
def kv_elements(past_kv):
    total = 0
    for i in range(len(past_kv)):
        k, v = past_kv[i]                         # (B, KV_H, S, D)
        total += k.numel() + v.numel()
    return total


# ----- Memory pressure metric
def memory_pressure(past_kv, max_kv_elements):
    if past_kv is None or len(past_kv) == 0:
        return 0.0
    return kv_elements(past_kv) / max_kv_elements


In [ ]:
def gating_router(layer_idx, token_imp, attn_entropy, mem_pressure):
    """
    Returns one of:
    - "full"
    - "layer_sparse"
    - "head"
    - "topk"
    """

    TOTAL_LAYERS = 16           # adjust if different
    EARLY_PROTECT = 10          # protect first 10 layers
    LATE_START = TOTAL_LAYERS - 4  # last 4 layers only

    # 1) High memory pressure → aggressive pruning
    if mem_pressure > 0.5:

        # Only allow layer_sparse in very deep layers
        if layer_idx >= LATE_START:
            return "layer_sparse"

        # Otherwise prefer head pruning
        return "head"

    # 2) Medium pressure → structure-aware
    if mem_pressure > 0.3:

        # Never drop early layers
        if layer_idx < EARLY_PROTECT:
            return "head"

        if attn_entropy > 2.0:
            return "topk"
        else:
            return "head"

    # 3) Low pressure → importance-based
    if token_imp < 0.6:
        return "topk"

    # 4) Otherwise keep everything
    return "full"

In [ ]:
import torch
from transformers.cache_utils import DynamicCache
from collections import Counter

# ======================================================
# Helper: KV element counter (SAFE FOR YOUR HF VERSION)
# ======================================================
def kv_elements(past_kv):
    total = 0
    for layer in list(past_kv):      # DynamicCache is iterable
        k = layer[0]                 # (k, v, extra)
        v = layer[1]
        total += k.numel() + v.numel()
    return total


def memory_pressure(past_kv, max_kv_elements):
    if past_kv.get_seq_length() == 0:
        return 0.0
    return kv_elements(past_kv) / max_kv_elements


# ======================================================
# INIT
# ======================================================
inputs = tokenizer("The future of AI is", return_tensors="pt").to(model.device)
input_ids = inputs["input_ids"]

analysis_cache = DynamicCache()
exec_cache = DynamicCache()

MAX_KV_ELEMENTS = 600_000
PRINT_LAYERS = 5

experts = {
    "full": FullCacheExpert(),
    "topk": TopKTokenExpert(K=4),
    "head": HeadSpecificExpert(keep_heads=[0], window=1),
    "layer_sparse": LayerSparseExpert(drop_layers=[]),
}

# ======================================================
# MAIN LOOP
# ======================================================
for step in range(15):

    # ======================================================
    # PASS 1 — ANALYSIS
    # ======================================================
    analysis_input = input_ids if analysis_cache.get_seq_length() == 0 else input_ids[:, -1:]

    with torch.no_grad():
        analysis_out = model(
            input_ids=analysis_input,
            past_key_values=analysis_cache,
            use_cache=True,
            output_attentions=True
        )

    routing = []
    mem_p = memory_pressure(exec_cache, MAX_KV_ELEMENTS)

    for layer_idx, attn in enumerate(analysis_out.attentions):
        t_imp = token_importance(attn)
        ent = attention_entropy(attn)

        expert_name = gating_router(layer_idx, t_imp, ent, mem_p)
        routing.append(expert_name)

        if layer_idx < PRINT_LAYERS:
            print(
                f"Step {step:02d} | Layer {layer_idx:02d} "
                f"| imp={t_imp:.2f} ent={ent:.2f} → {expert_name}"
            )

    print(f"Step {step:02d} routing summary:", dict(Counter(routing)))

    # 🔴 CRITICAL FIX:
    # analysis_out.past_key_values already contains full updated cache.
    analysis_cache = analysis_out.past_key_values

    # ======================================================
    # PASS 2 — EXECUTION (MoE-KV)
    # ======================================================
    exec_input = input_ids if exec_cache.get_seq_length() == 0 else input_ids[:, -1:]

    with torch.no_grad():
        exec_out = model(
            input_ids=exec_input,
            past_key_values=exec_cache,
            use_cache=True,
            output_attentions=False
        )

    next_token = exec_out.logits[:, -1].argmax(dim=-1, keepdim=True)

    exec_past_layers = list(exec_cache)
    new_exec_cache = DynamicCache()

    new_layers = list(exec_out.past_key_values)

    for layer_idx, layer in enumerate(new_layers):

        k_full = layer[0]   # (k, v, extra)
        v_full = layer[1]

        expert = experts[routing[layer_idx]]

        if layer_idx < len(exec_past_layers):
            past_k = exec_past_layers[layer_idx][0]
            past_v = exec_past_layers[layer_idx][1]
            past_layer_kv = (past_k, past_v)
        else:
            past_layer_kv = None

        k_new, v_new = expert.update(
            past_layer_kv,
            (k_full, v_full),
            layer_idx
        )

        # new_exec_cache is empty → safe to update with full KV
        new_exec_cache.update(k_new, v_new, layer_idx)

    exec_cache = new_exec_cache
    input_ids = torch.cat([input_ids, next_token], dim=-1)

Step 00 | Layer 00 | imp=0.72 ent=1.15 → full
Step 00 | Layer 01 | imp=0.80 ent=0.86 → full
Step 00 | Layer 02 | imp=0.85 ent=0.63 → full
Step 00 | Layer 03 | imp=0.71 ent=1.04 → full
Step 00 | Layer 04 | imp=0.71 ent=1.10 → full
Step 00 routing summary: {'full': 16}
Step 01 | Layer 00 | imp=0.72 ent=1.16 → full
Step 01 | Layer 01 | imp=0.84 ent=0.76 → full
Step 01 | Layer 02 | imp=0.92 ent=0.42 → full
Step 01 | Layer 03 | imp=0.79 ent=0.87 → full
Step 01 | Layer 04 | imp=0.76 ent=0.99 → full
Step 01 routing summary: {'full': 16}
Step 02 | Layer 00 | imp=0.74 ent=1.15 → full
Step 02 | Layer 01 | imp=0.85 ent=0.71 → full
Step 02 | Layer 02 | imp=0.86 ent=0.65 → full
Step 02 | Layer 03 | imp=0.76 ent=1.04 → full
Step 02 | Layer 04 | imp=0.75 ent=1.03 → full
Step 02 routing summary: {'full': 16}
Step 03 | Layer 00 | imp=0.69 ent=1.43 → full
Step 03 | Layer 01 | imp=0.83 ent=0.82 → full
Step 03 | Layer 02 | imp=0.85 ent=0.72 → full
Step 03 | Layer 03 | imp=0.69 ent=1.26 → full
Step 03 | La

In [ ]:
# ======================================================
# Unified Effective KV Benchmark (All Methods + MoE)
# ======================================================

from collections import Counter
import numpy as np
import time
import torch
from transformers.cache_utils import DynamicCache


# -----------------------------
# Metrics
# -----------------------------
def kv_elements(past_kv):
    total = 0
    for layer in list(past_kv):
        k, v = layer[0], layer[1]
        total += k.numel() + v.numel()
    return total


def kv_nonzero_ratio(past_kv):
    total = 0
    nonzero = 0
    for layer in list(past_kv):
        k, v = layer[0], layer[1]
        total += k.numel() + v.numel()
        nonzero += (k != 0).sum().item()
        nonzero += (v != 0).sum().item()
    return nonzero / total


def memory_pressure(past_kv, max_kv_elements):
    if past_kv.get_seq_length() == 0:
        return 0.0
    return kv_elements(past_kv) / max_kv_elements


# -----------------------------
# Single Expert Runner
# -----------------------------
def run_expert(prompt, expert=None, steps=40):

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_ids = inputs["input_ids"]

    cache = DynamicCache()

    t0 = time.time()

    for step in range(steps):

        step_in = input_ids if cache.get_seq_length() == 0 else input_ids[:, -1:]

        with torch.no_grad():
            out = model(input_ids=step_in, past_key_values=cache, use_cache=True)

        layers = list(out.past_key_values)
        new_cache = DynamicCache()

        for layer_idx, layer in enumerate(layers):
            k_full, v_full = layer[0], layer[1]

            if expert is None:
                k_new, v_new = k_full, v_full
            else:
                k_new, v_new = expert.update(None, (k_full, v_full), layer_idx)

            new_cache.update(k_new, v_new, layer_idx)

        cache = new_cache

        next_token = out.logits[:, -1].argmax(dim=-1, keepdim=True)
        input_ids = torch.cat([input_ids, next_token], dim=-1)

    t1 = time.time()

    total_kv = kv_elements(cache)
    effective_kv = total_kv * kv_nonzero_ratio(cache)

    return {
        "total": total_kv,
        "effective": effective_kv,
        "time": t1 - t0,
        "tok_s": steps / (t1 - t0),
    }


# -----------------------------
# MoE Runner
# -----------------------------
def run_moe(prompt, steps=40, MAX_KV_ELEMENTS=600_000):

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_ids = inputs["input_ids"]

    analysis_cache = DynamicCache()
    exec_cache = DynamicCache()

    routing_counts = Counter()

    t0 = time.time()

    for step in range(steps):

        # ---- Analysis pass
        analysis_in = input_ids if analysis_cache.get_seq_length() == 0 else input_ids[:, -1:]

        with torch.no_grad():
            analysis_out = model(
                input_ids=analysis_in,
                past_key_values=analysis_cache,
                use_cache=True,
                output_attentions=True
            )

        mem_p = memory_pressure(exec_cache, MAX_KV_ELEMENTS)

        routing = []
        for layer_idx, attn in enumerate(analysis_out.attentions):
            t_imp = token_importance(attn)
            ent = attention_entropy(attn)
            expert_name = gating_router(layer_idx, t_imp, ent, mem_p)
            routing.append(expert_name)
            routing_counts[expert_name] += 1

        analysis_cache = analysis_out.past_key_values

        # ---- Execution pass
        exec_in = input_ids if exec_cache.get_seq_length() == 0 else input_ids[:, -1:]

        with torch.no_grad():
            exec_out = model(input_ids=exec_in, past_key_values=exec_cache, use_cache=True)

        new_exec_cache = DynamicCache()
        layers = list(exec_out.past_key_values)

        for layer_idx, layer in enumerate(layers):
            k_full, v_full = layer[0], layer[1]
            expert = experts[routing[layer_idx]]
            k_new, v_new = expert.update(None, (k_full, v_full), layer_idx)
            new_exec_cache.update(k_new, v_new, layer_idx)

        exec_cache = new_exec_cache

        next_token = exec_out.logits[:, -1].argmax(dim=-1, keepdim=True)
        input_ids = torch.cat([input_ids, next_token], dim=-1)

    t1 = time.time()

    total_kv = kv_elements(exec_cache)
    effective_kv = total_kv * kv_nonzero_ratio(exec_cache)

    return {
        "total": total_kv,
        "effective": effective_kv,
        "time": t1 - t0,
        "tok_s": steps / (t1 - t0),
        "routing": dict(routing_counts),
    }


# ======================================================
# RUN ALL METHODS
# ======================================================

prompt = "The future of AI is"

baseline = run_expert(prompt, expert=None)
head = run_expert(prompt, expert=HeadSpecificExpert(keep_heads=[0], window=1))
topk = run_expert(prompt, expert=TopKTokenExpert(K=4))
layer_sparse = run_expert(prompt, expert=LayerSparseExpert(drop_layers=[0,2,4,6]))
moe = run_moe(prompt)

methods = {
    "Baseline": baseline,
    "Head": head,
    "TopK": topk,
    "LayerSparse": layer_sparse,
    "MoE": moe
}

print("\n================ FULL EFFECTIVE KV COMPARISON ================\n")

baseline_eff = baseline["effective"]

for name, r in methods.items():
    reduction = (baseline_eff - r["effective"]) / baseline_eff * 100
    print(f"{name:12} | "
          f"TotalKV: {r['total']:,.0f} | "
          f"EffectiveKV: {int(r['effective']):,} | "
          f"Reduction: {reduction:6.2f}% | "
          f"Time: {r['time']:.2f}s | "
          f"Tok/s: {r['tok_s']:.2f}")

print("\nMoE Routing Distribution:")
print(moe["routing"])


================ FULL EFFECTIVE KV COMPARISON ================

Baseline     | TotalKV: 737,280 | EffectiveKV: 737,266 | Reduction:   0.00% | Time: 1.65s | Tok/s: 24.30
Head         | TotalKV: 737,280 | EffectiveKV: 106,493 | Reduction:  85.56% | Time: 2.58s | Tok/s: 15.48
TopK         | TotalKV: 737,280 | EffectiveKV: 65,536 | Reduction:  91.11% | Time: 2.00s | Tok/s: 19.99
LayerSparse  | TotalKV: 737,280 | EffectiveKV: 552,951 | Reduction:  25.00% | Time: 1.88s | Tok/s: 21.30
MoE          | TotalKV: 737,280 | EffectiveKV: 198,910 | Reduction:  73.02% | Time: 4.57s | Tok/s: 8.75

MoE Routing Distribution:
{'full': 94, 'topk': 2, 'head': 440, 'layer_sparse': 104}


In [ ]:
from collections import Counter
import numpy as np
import time
import torch
from transformers.cache_utils import DynamicCache

def kv_elements(past_kv):
    total = 0
    for layer in list(past_kv):
        k, v = layer[0], layer[1]
        total += k.numel() + v.numel()
    return total

def kv_nonzero_ratio(past_kv):
    total = 0
    nonzero = 0
    for layer in list(past_kv):
        k, v = layer[0], layer[1]
        total += k.numel() + v.numel()
        nonzero += (k != 0).sum().item()
        nonzero += (v != 0).sum().item()
    return nonzero / total

def memory_pressure(past_kv, max_kv_elements):
    if past_kv.get_seq_length() == 0:
        return 0.0
    return kv_elements(past_kv) / max_kv_elements
def run_baseline(prompt, steps=40):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_ids = inputs["input_ids"]

    cache = DynamicCache()
    sparsity_log = []

    t0 = time.time()
    for _ in range(steps):
        step_in = input_ids if cache.get_seq_length() == 0 else input_ids[:, -1:]
        with torch.no_grad():
            out = model(input_ids=step_in, past_key_values=cache, use_cache=True)
        cache = out.past_key_values
        sparsity_log.append(1 - kv_nonzero_ratio(cache))

        next_token = out.logits[:, -1].argmax(dim=-1, keepdim=True)
        input_ids = torch.cat([input_ids, next_token], dim=-1)

    t1 = time.time()
    return {
        "time": t1 - t0,
        "avg_sparsity": float(np.mean(sparsity_log)),
        "text": tokenizer.decode(input_ids[0]),
    }


def run_moe(prompt, steps=40, MAX_KV_ELEMENTS=600_000):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_ids = inputs["input_ids"]

    analysis_cache = DynamicCache()
    exec_cache = DynamicCache()

    routing_counts = Counter()
    sparsity_log = []

    t0 = time.time()
    for step in range(steps):
        # --- analysis pass
        analysis_in = input_ids if analysis_cache.get_seq_length() == 0 else input_ids[:, -1:]
        with torch.no_grad():
            analysis_out = model(
                input_ids=analysis_in,
                past_key_values=analysis_cache,
                use_cache=True,
                output_attentions=True
            )

        mem_p = memory_pressure(exec_cache, MAX_KV_ELEMENTS)

        routing = []
        for layer_idx, attn in enumerate(analysis_out.attentions):
            t_imp = token_importance(attn)
            ent = attention_entropy(attn)
            expert_name = gating_router(layer_idx, t_imp, ent, mem_p)
            routing.append(expert_name)
            routing_counts[expert_name] += 1

        analysis_cache = analysis_out.past_key_values

        # --- execution pass
        exec_in = input_ids if exec_cache.get_seq_length() == 0 else input_ids[:, -1:]
        with torch.no_grad():
            exec_out = model(input_ids=exec_in, past_key_values=exec_cache, use_cache=True)

        new_exec_cache = DynamicCache()
        layers = list(exec_out.past_key_values)

        for layer_idx, layer in enumerate(layers):
            k_full, v_full = layer[0], layer[1]
            expert = experts[routing[layer_idx]]
            k_new, v_new = expert.update(None, (k_full, v_full), layer_idx)
            new_exec_cache.update(k_new, v_new, layer_idx)

        exec_cache = new_exec_cache
        sparsity_log.append(1 - kv_nonzero_ratio(exec_cache))

        next_token = exec_out.logits[:, -1].argmax(dim=-1, keepdim=True)
        input_ids = torch.cat([input_ids, next_token], dim=-1)

    t1 = time.time()
    return {
        "time": t1 - t0,
        "avg_sparsity": float(np.mean(sparsity_log)),
        "routing": dict(routing_counts),
        "text": tokenizer.decode(input_ids[0]),
    }
prompt = "Who is epstein?"

baseline = run_baseline(prompt, steps=40)
moe = run_moe(prompt, steps=40, MAX_KV_ELEMENTS=600_000)

print("\n===== BASELINE =====")
print("Time:", baseline["time"])
print("Avg sparsity:", baseline["avg_sparsity"] * 100, "%")

print("\n===== MOE =====")
print("Time:", moe["time"])
print("Avg sparsity:", moe["avg_sparsity"] * 100, "%")
print("Routing counts:", moe["routing"])

print("\nBaseline text (first 200 chars):")
print(baseline["text"][:400])

print("\nMoE text (first 200 chars):")
print(moe["text"][:400])


===== BASELINE =====
Time: 1.8000383377075195
Avg sparsity: 0.0028979526378039577 %

===== MOE =====
Time: 4.221970796585083
Avg sparsity: 67.62562167118074 %
Routing counts: {'full': 90, 'topk': 6, 'head': 440, 'layer_sparse': 104}

Baseline text (first 200 chars):
<|begin_of_text|>Who is epstein? Jeffrey Epstein was a convicted sex offender who was accused of running a sex trafficking ring and was arrested in 2019. He died in his cell in August 2019 while awaiting trial.
Epstein

MoE text (first 200 chars):
<|begin_of_text|>Who is epstein? Jeffrey Epstein was a wealthy businessman who is. and the devil's work.

The final answer, however, as this is nonsense.

-1, }
-1
:Question
-1 is not


In [ ]:
# ============================================================
# INTERACTIVE FULL ANSWER + CACHE INSPECTION
# ============================================================

import torch
from transformers.cache_utils import DynamicCache
from collections import Counter

# ---------------------------
# Utility Functions
# ---------------------------

def kv_elements(past_kv):
    total = 0
    for layer in list(past_kv):
        k, v = layer[0], layer[1]
        total += k.numel() + v.numel()
    return total


def kv_nonzero_ratio(past_kv):
    total = 0
    nonzero = 0
    for layer in list(past_kv):
        k, v = layer[0], layer[1]
        total += k.numel() + v.numel()
        nonzero += (k != 0).sum().item()
        nonzero += (v != 0).sum().item()
    return nonzero / total if total > 0 else 0.0


def print_cache_stats(name, cache):
    if cache.get_seq_length() == 0:
        print(f"{name}: Empty")
        return

    total = kv_elements(cache)
    sparsity = 1 - kv_nonzero_ratio(cache)

    print(f"{name} | SeqLen: {cache.get_seq_length()} "
          f"| TotalKV: {total:,} "
          f"| Sparsity: {sparsity*100:.2f}%")

# ---------------------------
# Initialize caches
# ---------------------------

baseline_cache = DynamicCache()
analysis_cache = DynamicCache()
exec_cache = DynamicCache()

MAX_KV_ELEMENTS = 600_000
GEN_STEPS = 40

print("\nInteractive Mode (type 'exit' to stop)\n")

while True:

    prompt = input("\nYour prompt: ")

    if prompt.lower() == "exit":
        break

    # ======================================================
    # BASELINE GENERATION
    # ======================================================

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_ids_base = inputs["input_ids"]

    for _ in range(GEN_STEPS):

        step_in = input_ids_base if baseline_cache.get_seq_length() == 0 else input_ids_base[:, -1:]

        with torch.no_grad():
            out_base = model(
                input_ids=step_in,
                past_key_values=baseline_cache,
                use_cache=True
            )

        baseline_cache = out_base.past_key_values
        next_token = out_base.logits[:, -1].argmax(dim=-1, keepdim=True)
        input_ids_base = torch.cat([input_ids_base, next_token], dim=-1)

    baseline_text = tokenizer.decode(input_ids_base[0])

    # ======================================================
    # MOE GENERATION
    # ======================================================

    input_ids_moe = inputs["input_ids"]
    routing_counter = Counter()

    for _ in range(GEN_STEPS):

        # ---- Analysis pass
        analysis_in = input_ids_moe if analysis_cache.get_seq_length() == 0 else input_ids_moe[:, -1:]

        with torch.no_grad():
            analysis_out = model(
                input_ids=analysis_in,
                past_key_values=analysis_cache,
                use_cache=True,
                output_attentions=True
            )

        mem_p = kv_elements(exec_cache) / MAX_KV_ELEMENTS if exec_cache.get_seq_length() > 0 else 0.0

        routing = []
        for layer_idx, attn in enumerate(analysis_out.attentions):
            t_imp = token_importance(attn)
            ent = attention_entropy(attn)
            expert_name = gating_router(layer_idx, t_imp, ent, mem_p)
            routing.append(expert_name)
            routing_counter[expert_name] += 1

        analysis_cache = analysis_out.past_key_values

        # ---- Execution pass
        exec_in = input_ids_moe if exec_cache.get_seq_length() == 0 else input_ids_moe[:, -1:]

        with torch.no_grad():
            exec_out = model(
                input_ids=exec_in,
                past_key_values=exec_cache,
                use_cache=True
            )

        new_exec_cache = DynamicCache()
        layers = list(exec_out.past_key_values)

        for layer_idx, layer in enumerate(layers):
            k_full, v_full = layer[0], layer[1]
            expert = experts[routing[layer_idx]]
            k_new, v_new = expert.update(None, (k_full, v_full), layer_idx)
            new_exec_cache.update(k_new, v_new, layer_idx)

        exec_cache = new_exec_cache

        next_token = exec_out.logits[:, -1].argmax(dim=-1, keepdim=True)
        input_ids_moe = torch.cat([input_ids_moe, next_token], dim=-1)

    moe_text = tokenizer.decode(input_ids_moe[0])

    # ======================================================
    # PRINT RESULTS
    # ======================================================

    print("\n================ BASELINE ANSWER ================")
    print(baseline_text[:800])
    print_cache_stats("BaselineCache", baseline_cache)

    print("\n================ MOE ANSWER =====================")
    print(moe_text[:800])
    print_cache_stats("MoECache", exec_cache)

    print("Routing summary:", dict(routing_counter))


Interactive Mode (type 'exit' to stop)


================ BASELINE ANSWER ================
<|begin_of_text|>who is donauld trump? Donald Trump is an American businessman, television personality, and politician who served as the 45th President of the United States from 2017 to 2021. He is known for his business ac
BaselineCache | SeqLen: 47 | TotalKV: 770,048 | Sparsity: 0.00%

================ MOE ANSWER =====================
<|begin_of_text|>who is donauld trump? Donald Trump is an American businessman, and the more, the walled, the better, but not the best, but not the best, the. of the  and the best of the best
MoECache | SeqLen: 47 | TotalKV: 770,048 | Sparsity: 73.07%
Routing summary: {'full': 59, 'topk': 5, 'head': 464, 'layer_sparse': 112}

================ BASELINE ANSWER ================
<|begin_of_text|>who is epstein? Donald Trump is a businessman and real estate developer who has made his fortune through various ventures, including his family's real estate company, the Tru